# Exploração inicial do PublicHearingBR NLI

Este notebook explora o subconjunto de inferência em linguagem natural do PublicHearingBR. A unidade de análise não é a audiência inteira: cada exemplo combina uma opinião extraída, quatro trechos próximos da transcrição e uma anotação manual sobre a possibilidade de inferir a opinião a partir desses trechos.

As fontes são o [paper PublicHearingBR](../2410.07495v2.pdf) e a [dataset no Hugging Face](https://huggingface.co/datasets/unicamp-dl/PublicHearingBR).

> Escopo: usamos somente "PublicHearingBR_NLI.jsonl". O LDS, explorado no notebook anterior, é o corpus principal de sumarização e possui outro esquema. Aqui não treinamos nem avaliamos novos modelos; descrevemos os exemplos NLI e as verificações já disponibilizadas pelo dataset.

## 1. Download, carregamento e validação

O download seleciona explicitamente o JSONL NLI. Essa separação é necessária porque os arquivos LDS e NLI têm estruturas incompatíveis no repositório público.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

working_dir = Path.cwd().resolve()
project_root = working_dir if (working_dir / 'src').exists() else working_dir.parent
sys.path.insert(0, str(project_root / 'src'))

from ideias_em_rede.data.download import download_nli_dataset
from ideias_em_rede.data.load import load_jsonl

sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
dataset_path = download_nli_dataset()
record_frame = load_jsonl(dataset_path)

expected_columns = {'id', 'metadados_extraidos'}
assert len(record_frame) == 206, 'O NLI deve referenciar 206 audiências.'
assert set(record_frame.columns) == expected_columns, 'O esquema do NLI mudou.'
assert record_frame['id'].is_unique, 'Os IDs de audiência devem ser únicos.'

display(Markdown(f"Arquivo carregado: `{dataset_path.name}`<br>Registros de audiência: **{len(record_frame)}**<br>IDs únicos: **{record_frame['id'].nunique()}**"))

## 2. Estrutura da tarefa NLI

O campo "id" apenas referencia uma audiência do LDS. Os exemplos efetivos estão aninhados em "metadados_extraidos": cada pessoa pode ter várias opiniões, e cada opinião traz os chunks recuperados e as verificações de alucinação.

In [ ]:
record_skeleton = {
    'id': 1,
    'metadados_extraidos': {
        'assunto': '...',
        'tl_dr': '...',
        'envolvidos': [
            {
                'nome': '...',
                'cargo': '...',
                'opinioes': [
                    {
                        'opiniao': '...',
                        'chunks_proximos': ['...', '...', '...', '...'],
                        'verificacao_alucinacao': {
                            'verificacao_manual': True,
                            'prompt_1_<modelo>': {'alucinacao': True},
                        },
                    }
                ],
            }
        ],
    },
}

display(Markdown(f'```json\n{json.dumps(record_skeleton, ensure_ascii=False, indent=2)}\n```'))

field_frame = pd.DataFrame(
    [
        ('id', 'Referência à audiência do LDS.'),
        ('assunto', 'Tema extraído para a audiência.'),
        ('envolvidos', 'Pessoas e cargos extraídos pelo experimento.'),
        ('opiniao', 'Afirmação atribuída a uma pessoa pelo experimento.'),
        ('chunks_proximos', 'Trechos recuperados para verificar a opinião.'),
        ('verificacao_manual', 'Anotação manual de possível alucinação.'),
    ],
    columns=['field', 'meaning'],
)
display(field_frame)

In [ ]:
def count_words(text):
    return len(str(text).split())


def shorten_text(text, limit=360):
    normalized_text = ' '.join(str(text).split())
    return normalized_text if len(normalized_text) <= limit else f'{normalized_text[:limit - 3]}...'


def build_nli_frame(records):
    rows = []
    for record in records.to_dict('records'):
        extracted_metadata = record['metadados_extraidos']
        for person in extracted_metadata['envolvidos']:
            for opinion in person['opinioes']:
                verification = opinion['verificacao_alucinacao']
                automatic_verifications = {
                    key: value
                    for key, value in verification.items()
                    if key.startswith('prompt_')
                }
                automatic_predictions = {
                    key: value.get('alucinacao')
                    for key, value in automatic_verifications.items()
                }
                rows.append(
                    {
                        'hearing_id': record['id'],
                        'subject': extracted_metadata['assunto'],
                        'person_name': person['nome'],
                        'person_role': person['cargo'],
                        'opinion': opinion['opiniao'],
                        'context_chunks': opinion['chunks_proximos'],
                        'chunk_count': len(opinion['chunks_proximos']),
                        'manual_hallucination': verification['verificacao_manual'],
                        'automatic_verifications': automatic_verifications,
                        **automatic_predictions,
                    }
                )
    return pd.DataFrame(rows)


nli_frame = build_nli_frame(record_frame)
automatic_columns = sorted(
    column for column in nli_frame.columns if column.startswith('prompt_')
)
nli_frame['opinion_word_count'] = nli_frame['opinion'].map(count_words)
nli_frame['context_word_count'] = nli_frame['context_chunks'].map(
    lambda chunks: sum(count_words(chunk) for chunk in chunks)
)
nli_frame['manual_label'] = nli_frame['manual_hallucination'].map(
    {False: 'Inferível nos chunks', True: 'Possível alucinação'}
)

In [ ]:
assert len(nli_frame) == 4238, 'O NLI deve conter 4.238 opiniões.'
assert nli_frame['hearing_id'].nunique() == 206, 'O NLI deve cobrir 206 audiências.'
assert nli_frame['manual_hallucination'].notna().all(), 'Todo exemplo deve ter rótulo manual.'
assert nli_frame['chunk_count'].between(0, 4).all(), 'A quantidade de chunks mudou.'

chunk_count_frame = (
    nli_frame['chunk_count']
    .value_counts()
    .sort_index()
    .rename_axis('chunk_count')
    .reset_index(name='opinion_count')
)

display(Markdown(f"Opiniões NLI: **{len(nli_frame):,}**<br>Verificações automáticas: **{len(automatic_columns)}**<br>Opiniões com quatro chunks: **{nli_frame['chunk_count'].eq(4).sum():,}**".replace(',', '.')))
display(chunk_count_frame)
display(nli_frame[['hearing_id', 'person_name', 'opinion', 'chunk_count', 'manual_label']].head(5))

In [ ]:
missing_context_frame = nli_frame.loc[
    nli_frame['chunk_count'].eq(0),
    [
        'hearing_id',
        'subject',
        'person_name',
        'person_role',
        'opinion',
        'manual_label',
        'chunk_count',
    ],
]

display(missing_context_frame)

## 3. Dois exemplos práticos

Cada bloco abaixo mostra uma opinião, os quatro trechos usados para julgá-la, a anotação manual e as respostas automáticas. "Possível alucinação" significa que a opinião não pôde ser inferida desses chunks na anotação manual. Contudo, isso não prova, por si só, que ela seja falsa em toda a transcrição.

In [ ]:
prompt_strategy_frame = pd.DataFrame(
    [
        ('Prompt 1', 'Pergunta se toda a opinião pode ser inferida e solicita decisão em JSON com explicação.'),
        ('Prompt 2', 'Acrescenta a identificação de trechos de apoio e exige que toda a opinião seja diretamente sustentada.'),
        ('Prompt 3', 'Torna o critério mais estrito: em caso de dúvida ou suporte parcial, a opinião não é inferível.'),
    ],
    columns=['prompt', 'estratégia'],
)
display(Markdown('### Estratégias de prompt usadas no experimento'))
display(prompt_strategy_frame)
display(Markdown('O JSON disponibiliza as respostas identificadas por prompt e modelo. A redação integral dos prompts está nas Figuras 8, 9 e 10 do paper.'))

example_model_names = {
    'gpt-4o-mini-2024-07-18': 'GPT-4o mini',
    'gpt-4o-2024-08-06': 'GPT-4o',
    'deepseek-chat': 'DeepSeek-V3',
    'sabia-3.1-2025-05-08': 'Sabiá-3.1',
}
example_ids = [1, 2]

for hearing_id in example_ids:
    example_row = nli_frame.loc[nli_frame['hearing_id'].eq(hearing_id)].iloc[0]
    display(Markdown(f"### Audiência {hearing_id}"))
    display(Markdown(f"**Assunto:** {example_row['subject']}<br>**Pessoa:** {example_row['person_name']} - {example_row['person_role']}<br>**Anotação manual:** {example_row['manual_label']}"))
    display(pd.DataFrame({'opinion': [example_row['opinion']]}))
    chunk_frame = pd.DataFrame(
        {
            'chunk': range(1, example_row['chunk_count'] + 1),
            'text_preview': [
                shorten_text(chunk)
                for chunk in example_row['context_chunks']
            ],
        }
    )
    display(chunk_frame)
    prediction_rows = []
    for configuration, result in example_row['automatic_verifications'].items():
        prompt_number = configuration.split('_')[1]
        model_key = configuration.split('_', 2)[2]
        supporting_texts = result.get('trechos_para_basear_analise', [])
        prediction_rows.append(
            {
                'model': example_model_names[model_key],
                'prompt': f'Prompt {prompt_number}',
                'automatic_label': (
                    'Possível alucinação'
                    if result.get('alucinacao')
                    else 'Inferível nos chunks'
                ),
                'supporting_text_count': len(supporting_texts),
                'explanation_preview': shorten_text(
                    result.get('explicacao', ''),
                    limit=360,
                ),
            }
        )
    prediction_frame = pd.DataFrame(prediction_rows)
    display(Markdown('#### Respostas automáticas para a mesma opinião e contexto'))
    display(prediction_frame)

## 4. Métricas do conjunto

As medidas abaixo descrevem o nível de opinião: o tamanho da afirmação, o tamanho total do contexto recuperado e a quantidade de opiniões associadas a cada audiência.

In [ ]:
hearing_summary_frame = (
    nli_frame
    .groupby('hearing_id')
    .agg(
        opinion_count=('opinion', 'size'),
        participant_count=('person_name', 'nunique'),
    )
    .reset_index()
)

metric_frame = nli_frame[[
    'opinion_word_count',
    'context_word_count',
]].describe().T
metric_frame = metric_frame[[
    'mean', 'std', 'min', '25%', '50%', '75%', 'max'
]].rename(
    columns={'50%': 'median'}
)

display(metric_frame.round(1))
display(hearing_summary_frame[['opinion_count', 'participant_count']].describe().T.round(1))

In [ ]:
label_count_frame = (
    nli_frame['manual_label']
    .value_counts()
    .reindex(['Inferível nos chunks', 'Possível alucinação'])
    .rename_axis('manual_label')
    .reset_index(name='opinion_count')
)
label_count_frame['percentage'] = label_count_frame['opinion_count'] / len(nli_frame) * 100

figure, axis = plt.subplots(figsize=(8, 5))
bars = axis.bar(
    label_count_frame['manual_label'],
    label_count_frame['opinion_count'],
    color=['#4c956c', '#c1666b'],
    width=0.65,
)
axis.bar_label(
    bars,
    labels=[
        f'{count:,} ({percentage:.1f}%)'.replace(',', '.')
        for count, percentage in zip(
            label_count_frame['opinion_count'],
            label_count_frame['percentage'],
        )
    ],
    padding=3,
)
axis.set_title('Rótulos manuais de inferibilidade')
axis.set_xlabel('')
axis.set_ylabel('Opiniões')
axis.set_ylim(0, label_count_frame['opinion_count'].max() * 1.12)
plt.show()

In [ ]:
figure, axis = plt.subplots(figsize=(10, 5))
sns.histplot(
    data=hearing_summary_frame,
    x='opinion_count',
    bins=18,
    color='#6a4c93',
    edgecolor='white',
    ax=axis,
)
axis.set_title('Opiniões NLI por audiência')
axis.set_xlabel('Opiniões extraídas para a audiência')
axis.set_ylabel('Audiências')
plt.show()

In [ ]:
length_frame = nli_frame.melt(
    id_vars='manual_label',
    value_vars=['opinion_word_count', 'context_word_count'],
    var_name='text_type',
    value_name='word_count',
)
length_frame['text_type'] = length_frame['text_type'].map(
    {
        'opinion_word_count': 'Opinião',
        'context_word_count': 'Quatro chunks',
    }
)

figure, axis = plt.subplots(figsize=(10, 6))
sns.boxplot(
    data=length_frame,
    x='text_type',
    y='word_count',
    hue='manual_label',
    palette=['#4c956c', '#c1666b'],
    showfliers=False,
    ax=axis,
)
axis.set_title('Extensão da opinião e do contexto por rótulo manual')
axis.set_xlabel('')
axis.set_ylabel('Palavras')
axis.legend(title='Rótulo manual')
plt.show()

## 5. Verificações automáticas presentes no arquivo

O dataset inclui 12 combinações de modelo e prompt. A comparação abaixo usa o rótulo manual como referência e não representa uma nova avaliação executada neste projeto. A taxa no eixo Y é a proporção de possíveis alucinações identificadas; o eixo X é a proporção de opiniões inferíveis classificadas incorretamente como alucinação.

In [ ]:
model_names = {
    'gpt-4o-mini-2024-07-18': 'GPT-4o mini',
    'gpt-4o-2024-08-06': 'GPT-4o',
    'deepseek-chat': 'DeepSeek-V3',
    'sabia-3.1-2025-05-08': 'Sabiá-3.1',
}
configuration_rows = []

for column in automatic_columns:
    prediction_frame = nli_frame[['manual_hallucination', column]].dropna()
    prompt_number = column.split('_')[1]
    model_key = column.split('_', 2)[2]
    hallucination_frame = prediction_frame.loc[
        prediction_frame['manual_hallucination']
    ]
    valid_frame = prediction_frame.loc[
        ~prediction_frame['manual_hallucination']
    ]
    configuration_rows.append(
        {
            'configuration': column,
            'model': model_names[model_key],
            'prompt': f'Prompt {prompt_number}',
            'true_positive_rate': hallucination_frame[column].mean() * 100,
            'false_positive_rate': valid_frame[column].mean() * 100,
        }
    )

configuration_frame = pd.DataFrame(configuration_rows)
display(
    configuration_frame.sort_values(['model', 'prompt']).round(1)
)

In [ ]:
model_order = list(configuration_frame['model'].drop_duplicates())
prompt_order = list(configuration_frame['prompt'].drop_duplicates())
model_palette = dict(zip(model_order, sns.color_palette('deep', n_colors=len(model_order))))
prompt_markers = dict(zip(prompt_order, ['o', 's', 'X']))
zoom_frame = configuration_frame.loc[
    configuration_frame['false_positive_rate'].between(11.5, 12.5)
    & configuration_frame['true_positive_rate'].between(80.8, 82.6)
].copy()

figure, (axis, zoom_axis, legend_axis) = plt.subplots(
    ncols=3,
    figsize=(17, 6),
    gridspec_kw={'width_ratios': [3.4, 1.25, 1.05]},
    constrained_layout=True,
)
sns.scatterplot(
    data=configuration_frame,
    x='false_positive_rate',
    y='true_positive_rate',
    hue='model',
    style='prompt',
    hue_order=model_order,
    style_order=prompt_order,
    palette=model_palette,
    markers=prompt_markers,
    s=110,
    ax=axis,
)
legend_handles, legend_labels = axis.get_legend_handles_labels()
axis.legend_.remove()
for row in configuration_frame.itertuples():
    axis.annotate(
        row.prompt.replace('Prompt ', 'P'),
        (row.false_positive_rate, row.true_positive_rate),
        xytext=(5, 5),
        textcoords='offset points',
        fontsize=8,
    )
axis.set_title('Troca entre detecção e falsos positivos')
axis.set_xlabel('Opiniões inferíveis marcadas como alucinação (%)')
axis.set_ylabel('Possíveis alucinações identificadas (%)')

sns.scatterplot(
    data=zoom_frame,
    x='false_positive_rate',
    y='true_positive_rate',
    hue='model',
    style='prompt',
    hue_order=model_order,
    style_order=prompt_order,
    palette=model_palette,
    markers=prompt_markers,
    s=110,
    legend=False,
    ax=zoom_axis,
)
for row in zoom_frame.itertuples():
    zoom_axis.annotate(
        f'{row.model}\n{row.prompt.replace("Prompt ", "P")}',
        (row.false_positive_rate, row.true_positive_rate),
        xytext=(5, 5),
        textcoords='offset points',
        fontsize=7,
    )
zoom_axis.set_title('Zoom da sobreposição', fontsize=10)
zoom_axis.set_xlabel('Falsos positivos (%)', fontsize=8)
zoom_axis.set_ylabel('Detecção (%)', fontsize=8)
zoom_axis.tick_params(labelsize=8)

legend_axis.set_axis_off()
legend_axis.legend(
    legend_handles,
    legend_labels,
    title='Configuração',
    loc='upper left',
)
plt.show()

## 6. Síntese

A síntese é calculada diretamente dos exemplos carregados. Ela descreve os limites da tarefa: a decisão manual se aplica aos chunks recuperados, e não substitui uma leitura integral da audiência.

In [ ]:
manual_hallucination_count = int(nli_frame['manual_hallucination'].sum())
manual_hallucination_rate = manual_hallucination_count / len(nli_frame) * 100
median_opinion_count = hearing_summary_frame['opinion_count'].median()
median_context_word_count = nli_frame['context_word_count'].median()

display(Markdown(
    f"O NLI reúne **{len(nli_frame):,}** opiniões de **{nli_frame['hearing_id'].nunique()}** audiências. ".replace(',', '.')
    + f"Há **{manual_hallucination_count:,}** exemplos marcados manualmente como possível alucinação ({manual_hallucination_rate:.1f}%). ".replace(',', '.')
    + f"A audiência mediana reúne **{median_opinion_count:.0f}** opiniões NLI, e o contexto mediano dos exemplos tem **{median_context_word_count:.0f}** palavras. "
    + "Esses rótulos avaliam se a opinião é sustentada pelos chunks recuperados; portanto, não equivalem automaticamente a uma verificação factual da audiência completa."
))